# core

> Using pyinstrument to profile FastHTML apps.

Sometimes when building FastHTML apps we run into performance bottlenecks. Figuring out what is slow can be challenging, especially when building apps with async components. That's where profiling tools like pyinstrument can help. Profilers are tools that show exactly how long each component of a project takes to run. Identifying slow parts of an app is the first step in figuring out how to make things run faster.

In [ ]:
#| default_exp core

In [ ]:
import os,time,pickle,inspect
from pathlib import Path
from fasthtml.common import *
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.middleware import Middleware
from functools import wraps
from collections import namedtuple
from contextlib import contextmanager
from fastcore.basics import patch
try: from rich import print
except: pass

from pyinstrument import Profiler
from pyinstrument.renderers import ConsoleRenderer, HTMLRenderer
from pyinstrument.session import Session

In [ ]:
from starlette.testclient import TestClient
from fastcore.all import *
from functools import partialmethod
from anyio import from_thread

In [ ]:
def get_trigger_name(): return os.environ.get('PYINSTRUMENT_TRIGGER', 'profile')

In [ ]:
class ProfileMiddleware(BaseHTTPMiddleware):
    "Add the power of pyinstrument to potentially every request."
    def __init__(self, app, save_dir=None):
        super().__init__(app)
        self.save_dir = Path(save_dir) if save_dir else None

    def _save(self, profiler):
        if not self.save_dir: return
        self.save_dir.mkdir(parents=True, exist_ok=True)
        with open(self.save_dir/f'{time.time():.0f}.pkl', 'wb') as f: pickle.dump(profiler.last_session, f)

    async def dispatch(self, request, call_next):
        trigger_name = get_trigger_name()
        should_profile = request.query_params.get(trigger_name, False)
        term = request.query_params.get("term", False)
        if should_profile:
            profiler = Profiler()
            profiler.start()
            response = await call_next(request)
            profiler.stop()
            self._save(profiler)
            if term:
                print(profiler.output_text())
                return response
            return HTMLResponse(profiler.output_html())
        return await call_next(request)

In [ ]:
@contextmanager
def profiling():
    p = Profiler()
    p.start()
    try: yield p
    finally: p.stop()

In [ ]:
def instrument(route_handler):
    "Replaces the route handler's output with pyinstrument results."
    @wraps(route_handler)
    async def _wrapper(*args, **kwargs):
        with profiling() as p: await maybe_await(route_handler(*args, **kwargs))
        return p.output_html()
    return _wrapper

In [ ]:
def slow(): time.sleep(0.01)

@instrument
def my_route(): slow()

res = await my_route()
assert 'slow' in res and 'pyinstrument' in res.lower()
res[:50]

'<!DOCTYPE html>\n            <html>\n            <he'

In [ ]:
def load_session(path):
    "Load a saved pyinstrument session and return a renderer-ready object."
    with open(path, 'rb') as f: return pickle.load(f)


In [ ]:
app, rt = fast_app()
app.add_middleware(ProfileMiddleware, save_dir='/tmp/profiles')
client = TestClient(app)

@rt
def index(): return Titled('Hello, profiler')

In [ ]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

In [ ]:
ProfileEntry = namedtuple('ProfileEntry', 'time func file line')

def _matches(fp, paths): return not paths or any(p in fp for p in paths)

def _walk_frames(frame, fn, **kwargs):
    fn(frame, **kwargs)
    for c in frame.children: _walk_frames(c, fn, **kwargs)

In [ ]:
@patch
def flat(self:Session, paths=None, n=20):
    "Aggregate self-time by function, optionally filtered by file path substrings."
    acc = {}
    def _acc(frame):
        fp,st = frame.file_path or '',frame.total_self_time
        if st > 0 and _matches(fp, paths):
            k = (frame.function, fp, frame.line_no)
            acc[k] = acc.get(k, 0) + st
    _walk_frames(self.root_frame(), _acc)
    res = sorted([ProfileEntry(t, f, fp, ln) for (f,fp,ln),t in acc.items()], key=lambda o: -o.time)
    return res[:n]

`flat` walks the frame tree and sums `total_self_time` per unique (function, file, line) tuple. Filter with `paths` to focus on specific codebases. Returns a sorted list of `ProfileEntry` named tuples.

In [ ]:
sess = load_session(sorted(Path('/tmp/profiles').glob('*.pkl'))[-1])
for e in sess.flat(n=5): print(f'{e.time*1000:7.1f}ms  {e.func}  {e.file}:{e.line}')

1.7ms  run_sync_in_worker_thread  
/Users/jhoward/aai-ws/.venv/lib/python3.12/site-packages/anyio/_backends/_asyncio.py:2459

1.7ms    :None

1.0ms  getcoroutinestate  
/Users/jhoward/.local/share/uv/python/cpython-3.12.0-macos-aarch64-none/lib/python3.12/inspect.py:1919

1.0ms    :None

In [ ]:
@patch
def callers(self:Session, func_name, paths=None, n=10):
    "Find which functions call `func_name` and how much time they contribute."
    acc = {}
    def _acc(frame, parent=None):
        fp = frame.file_path or ''
        if frame.function == func_name and frame.total_self_time > 0 and parent:
            pfp = parent.file_path or ''
            if _matches(pfp, paths):
                k = (parent.function, pfp, parent.line_no)
                acc[k] = acc.get(k, 0) + frame.total_self_time
        for c in frame.children: _acc(c, parent=frame)
    _acc(self.root_frame())
    res = sorted([ProfileEntry(t, f, fp, ln) for (f,fp,ln),t in acc.items()], key=lambda o: -o.time)
    return res[:n]

`callers` answers "who is calling this hot function?". For each occurrence of `func_name` with self-time, it attributes that time to the immediate parent frame.

In [ ]:
for e in sess.callers('getattr', n=5): print(f'{e.time*1000:7.1f}ms  {e.func}  {e.file}:{e.line}')

In [ ]:
@patch
def callees(self:Session, func_name, paths=None, n=10):
    "Find what `func_name` spends its time calling."
    acc = {}
    def _acc(frame, inside=False):
        fp = frame.file_path or ''
        if frame.function == func_name: inside = True
        if inside and frame.total_self_time > 0 and frame.function != func_name and _matches(fp, paths):
            k = (frame.function, fp, frame.line_no)
            acc[k] = acc.get(k, 0) + frame.total_self_time
        for c in frame.children: _acc(c, inside)
    _acc(self.root_frame())
    res = sorted([ProfileEntry(t, f, fp, ln) for (f,fp,ln),t in acc.items()], key=lambda o: -o.time)
    return res[:n]

`callees` answers "what does this function spend its time on?". It walks descendants of matching frames and aggregates their self-time.

In [ ]:
for e in sess.callees('getattr', n=5): print(f'{e.time*1000:7.1f}ms  {e.func}  {e.file}:{e.line}')

In [ ]:
@patch
def hot_paths(self:Session, paths=None, n=10, depth=8):
    "Top call stacks by cumulative time, filtered to frames matching `paths`."
    stacks = {}
    def _collect(frame, stack=()):
        fp = frame.file_path or ''
        if _matches(fp, paths): stack = stack + (f"{frame.function} {Path(fp).name}:{frame.line_no}",)
        if frame.total_self_time > 0 and stack:
            trimmed = stack[-depth:]
            stacks[trimmed] = stacks.get(trimmed, 0) + frame.total_self_time
        for c in frame.children: _collect(c, stack)
    _collect(self.root_frame())
    return sorted([(t, ' → '.join(s)) for s,t in stacks.items()], key=lambda o: -o[0])[:n]

`hot_paths` shows the most expensive call stacks, collapsed to only frames matching `paths`. The `depth` parameter limits stack depth to keep output readable.

In [ ]:
ps = ['solveit/', 'fasthtml/', 'fastcore/']
for t,s in sess.hot_paths(paths=ps, n=5): print(f'{t*1000:7.1f}ms  {s}')

3.4ms  _f core.py:669 → _wrap_call core.py:478 → _handle core.py:269

In [ ]:
def render_session(sess, text=True, show_all=False, short_mode=True):
    "Render a saved session as text or html."
    if text: return ConsoleRenderer(show_all=show_all, short_mode=short_mode).render(sess)
    return HTMLRenderer(show_all=show_all).render(sess)

## Tests

First, confirm that the view works normally

In [ ]:
assert 'Hello, profiler' in client.get('/').text

Now lets profile it! Or rather, check that it works.

In [ ]:
'pyinstrumentHTMLRenderer' in client.get('/?profile=1').text

True

Let's print to the terminal

In [ ]:
client.get('/?profile=1&term=1')

_     ._   __/__   _ _  _  _ _/_   Recorded: 13:41:32  Samples:  1
 /_//_/// /_\ / //_// / //_'/ //     Duration: 0.001     CPU time: 0.001
/   _/                      v5.1.2

Profile at /var/folders/51/b2_szf2945n072c0vj2cyty40000gn/T/ipykernel_92920/363051617.py:18

0.001 Handle._run  asyncio/events.py:82
`- 0.001 coro  starlette/middleware/base.py:139
      [9 frames hidden]  starlette
         0.001 app  starlette/routing.py:60
         `- 0.001 _f  ../fasthtml/fasthtml/core.py:669
            `- 0.001 _wrap_call  ../fasthtml/fasthtml/core.py:478
               `- 0.001 _handle  ../fasthtml/fasthtml/core.py:269
                  `- 0.001 run_in_threadpool  starlette/concurrency.py:30
                     `- 0.001 run_sync  anyio/to_thread.py:25
                        `- 0.001 AsyncIOBackend.run_sync_in_worker_thread  anyio/_backends/_asyncio.py:2459
                           `- 0.001   anyio/_backends/_asyncio.py

<Response [200 OK]>

In [ ]:
@rt
@instrument
def saxaphone(): return Titled('Play that sweet horn')
assert 'pyinstrumentHTMLRenderer' in client.get('/saxaphone').text

In [ ]:
@rt
@instrument
async def trombone(): return Titled('Async horn')
assert 'pyinstrumentHTMLRenderer' in client.get('/trombone').text

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()